# Case 2 — High Season Decisions at Viador Hotels

**Data-Driven Decisions in Practice · D3 Applications · SS 2026**

**Participants - Group H**

| Group member | Matrikel-Nr |
|---|---|
| David Heuer | 3122724 |
| Amelie Schafferhans |  |
| Alexander Puppe | 3152720 |

Clara Schwarz oversees two mid-sized hotels under the Viador brand — a **city** location and a **resort** location. She wants your help refining pricing and capacity decisions for the upcoming high season. Your findings will brief her revenue and operations team: **clarity of communication is key**. Visualisations encouraged. *Pricing strategies should always rely on realised bookings, not mere requests.*

This template provides a **structural scaffold**. Every decision (segmentation thresholds, buffer levels, seasonal rules, adaptive policy) is yours to make and defend.

### Method companions

| Notebook | Focus | Feeds |
|---|---|---|
| [1 · EDA](https://chrisflath.github.io/notebooks/d3ip-case2-rm-1-data.html) | Booking curves, fare-class structure, EDA bridge | Q1, Q3 |
| [2 · Dynamic control](https://chrisflath.github.io/notebooks/d3ip-case2-rm-2-dynamic.html) | Booking strips, revenue surface, S-index | Q4 baseline |
| [3 · Prediction models](https://chrisflath.github.io/notebooks/d3ip-case2-rm-3-prediction.html) | Cancellation classifier, calibration, $q^{\text{eff}}_t$ | Q2, Q4 advanced |

### How to use this file

- Each question is one section. Add as many cells as you need.
- Use the loader in §0; do not re-download per question.
- The final reflection section is **required**.

## §0 · Setup & data loader

The loader below is the same one used in the three companion notebooks. It downloads the public mirror of the Viador booking dataset and adds two helper columns (`booking_date`, `adr_per_adult`).

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/mpolinowski/hotel-booking-dataset"
    "/refs/heads/master/datasets/hotel_bookings.csv"
)




def load_bookings():
    df = pd.read_csv(DATA_URL)
    df["arrival_date"] = pd.to_datetime(
        df["arrival_date_year"].astype(str)
        + "-"
        + df["arrival_date_month"]
        + "-"
        + df["arrival_date_day_of_month"].astype(str),
        format="%Y-%B-%d",
        errors="coerce",
    )
    df["booking_date"] = df["arrival_date"] - pd.to_timedelta(
        df["lead_time"], unit="D"
    )
    df["adr_per_adult"] = df["adr"] / df["adults"].clip(lower=1)
    return df


bookings = load_bookings()
bookings.shape, bookings["arrival_date"].min(), bookings["arrival_date"].max()

: 

## §Q1 · Calibrate Advance Sale Policies

Clara offers discounted advance rates — but only up to a certain number of bookings. Selling too many in advance turns away spontaneous, higher-paying guests; selling too few leaves occupancy short.

- Using **realised stays**, identify meaningful guest segments for each hotel. *When* do different guest types book? *What* prices do they pay?
- Estimate how many rooms Clara could allocate to advance sales **for each property**. Focus on lead times and, if useful, guest mix.

*Methods reference: companion NB1 §3 (univariate), §4 (per-hotel), §8 (booking curves), §9 (canonical-booking cut tool — drives the advance/late split per hotel).*

Für beide Hotels soll in dieser Aufgabe berechnet werden, viel viele Zimmer man spät und teuer buchenden Gästen vorbehalten sollte.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.size":        11,
})

HOTEL_COLORS = {"City Hotel": "#1f77b4", "Resort Hotel": "#ff7f0e"}
HOTELS = ["City Hotel", "Resort Hotel"]

# Realised stays only — pricing decisions must rest on guests who actually arrived
realised = bookings[bookings["is_canceled"] == 0].copy()
realised["total_nights"] = (
    realised["stays_in_weekend_nights"] + realised["stays_in_week_nights"]
)

stay_df = realised[realised["total_nights"] > 0].copy()
stay_df["night_dates"] = stay_df.apply(
    lambda r: pd.date_range(r["arrival_date"], periods=int(r["total_nights"]), freq="D"),
    axis=1,
)
expanded = stay_df[["hotel", "night_dates"]].explode("night_dates")
daily_occ = (
    expanded.groupby(["hotel", "night_dates"])
    .size()
    .reset_index(name="overnight_stays")
)

print(f"Total bookings  : {len(bookings):,}")
print(f"Realised stays  : {len(realised):,}  ({len(realised)/len(bookings):.1%} of all bookings)")
print()
print(realised.groupby("hotel").size().rename("realised_stays").to_frame())


In [ ]:
# ── Figure 1: Lead-time distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
bins = np.arange(0, 368, 7)   # one bin per week

for ax, hotel in zip(axes, HOTELS):
    sub = realised[realised["hotel"] == hotel]["lead_time"]
    ax.hist(sub, bins=bins, color=HOTEL_COLORS[hotel], alpha=0.85,
            edgecolor="white", linewidth=0.3)

    med = sub.median()
    p25 = sub.quantile(0.25)
    p75 = sub.quantile(0.75)

    ax.axvline(p25, color="black", ls=":",  lw=1.4, label=f"Q1: {p25:.0f} d")
    ax.axvline(med, color="black", ls="--", lw=2.0, label=f"Median: {med:.0f} d")
    ax.axvline(p75, color="black", ls="-.", lw=1.4, label=f"Q3: {p75:.0f} d")

    ax.set_title(hotel, fontweight="bold", fontsize=13)
    ax.set_xlabel("Lead time (days before arrival)")
    ax.set_ylabel("Realised stays")
    ax.legend(fontsize=9)

fig.suptitle("Figure 1 · When Do Guests Book? — Lead-Time Distribution per Hotel",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("q1_fig1_lead_time.png", dpi=150, bbox_inches="tight")
plt.show()


**Abbildung 1 – Buchungsvorlauf pro Hotel**

Beide Hotels zeigen eine stark rechtsschiefe Verteilung: Je näher die Übernachtung heranrückt, desto mehr tatsächlich realisierte Buchungen werden getätigt. Beim Resort-Hotel wird aber noch kurzfristiger gebucht als beim City-Hotel. 25% der Buchungen gehen beim Resorthotel in den letzten fünf Tagen vor der Buchung ein, beim City-Hotel sind es immerhin 12 Tage. Bei der Medianbuchung sieht es ähnlich aus. Diese liegt beim Resort-Hotel 12 Tage näher an der Übernachtung als beim Cityhotel. Nur beim 75% Quantil gibt es eine Abweichung. Dieses wird beim Resorthotel schon früher im Jahr erreicht als beim City-Hotel. Die extremen Frühbucher sind also eher beim Resorthotel zu finden, die Mehrheit der Gäste bucht im Resorthotel allerdings später als im Cityhotel. Somit kann man entweder am 50% Quantil oder am 25% Quantil eine Grenze ziehen, und eine Aufteilung in Frühbucher und Last-Minute-Bucher erstellen.


In [ ]:
# ── Figure 2: ADR distribution per hotel ──────────────────────────────────────
adr_data = realised[realised["adr"].between(1, 500)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
bins = np.arange(0, 505, 10)  # 10€-Schritte

for ax, hotel in zip(axes, HOTELS):
    sub = adr_data[adr_data["hotel"] == hotel]["adr"]
    weights = np.ones(len(sub)) / len(sub) * 100  # Prozent statt Anzahl

    ax.hist(sub, bins=bins, weights=weights,
            color=HOTEL_COLORS[hotel], alpha=0.85,
            edgecolor="white", linewidth=0.3)

    for q, ls, lw, label in [
        (0.25, ":",  1.4, f"Q1: €{sub.quantile(0.25):.0f}"),
        (0.50, "--", 2.0, f"Median: €{sub.median():.0f}"),
        (0.75, "-.", 1.4, f"Q3: €{sub.quantile(0.75):.0f}"),
    ]:
        ax.axvline(sub.quantile(q), color="black", ls=ls, lw=lw, label=label)

    ax.set_title(hotel, fontweight="bold", fontsize=13)
    ax.set_xlabel("ADR (€ pro Nacht)")
    ax.set_ylabel("Anteil der Buchungen (%)")
    ax.legend(fontsize=9)

fig.suptitle("Figure 2 · ADR-Verteilung — Wie viel zahlen die Gäste?",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("q1_fig2_adr_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

for hotel in HOTELS:
    sub = adr_data[adr_data["hotel"] == hotel]["adr"]
    print(f"{hotel}: Median €{sub.median():.0f} | Q1 €{sub.quantile(0.25):.0f} | Q3 €{sub.quantile(0.75):.0f} | Max €{sub.max():.0f}")


**Abbildung 2 – ADR-Verteilung pro Hotel**

Dieses Diagramm zeig die Verteilung der Buchungen auf die bezahlten Preise. Interessant ist vor allem das 75% Quantil, da es alle teuren Buchungen anzeigt. Also diejenigen, für die wir am Ende Zimmer blocken wollen, um sicherzustellen, dass die gutbezahlenden Gäste auch sicher zimmer bekommen. Bei beiden Hotels liegt das 75% Quantil in einem ähnlichen Bereich, nämlich bei 120€ bzw. 127€.


In [ ]:
# ── Figure 3: Lead-time distribution for expensive bookings only ───────────────
ADR_THRESHOLDS = {"City Hotel": 127, "Resort Hotel": 120}

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
bins = np.arange(0, 368, 7)

for ax, hotel in zip(axes, HOTELS):
    threshold = ADR_THRESHOLDS[hotel]
    sub = realised[
        (realised["hotel"] == hotel) &
        (realised["adr"] > threshold)
    ]["lead_time"]

    ax.hist(sub, bins=bins, color=HOTEL_COLORS[hotel], alpha=0.85,
            edgecolor="white", linewidth=0.3)

    for q, ls, lw, label in [
        (0.25, ":",  1.4, f"Q1: {sub.quantile(0.25):.0f} d"),
        (0.50, "--", 2.0, f"Median: {sub.median():.0f} d"),
        (0.75, "-.", 1.4, f"Q3: {sub.quantile(0.75):.0f} d"),
    ]:
        ax.axvline(sub.quantile(q), color="black", ls=ls, lw=lw, label=label)

    ax.set_title(f"{hotel}  (ADR > €{threshold})", fontweight="bold", fontsize=13)
    ax.set_xlabel("Lead time (days before arrival)")
    ax.set_ylabel("Realised stays")
    ax.legend(fontsize=9)
    print(f"{hotel}: n={len(sub):,} | Median {sub.median():.0f} d | Q1 {sub.quantile(0.25):.0f} d | Q3 {sub.quantile(0.75):.0f} d")

fig.suptitle("Figure 3 · Buchungsvorlauf teurer Buchungen (oberstes Quartil)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("q1_fig3_expensive_lead_time.png", dpi=150, bbox_inches="tight")
plt.show()


**Abbildung 3 – Buchungsvorlauf teurer Buchungen**

Das Diagramm zeigt, wie viele Tage vor der Übernachtung die teuren Zimmer (75% Quantil) gebucht wurden. Hier ist beim City-Hotel ein klarer Unterschied zu Diagramm 1 zu sehen. In Diagramm 1 lag die Median-Lead-Time bei 50 Tagen, jetzt nur noch bei 35 Tagen. Je später die Buchung ist, desto höher ist ihr ADR beim City-Hotel. Hier lohnt es sich also prinzipiell, hochpreisige Zimmer länger für blockieren für viel Zahlende Gäste. Beim Resort Hotel ist es andersherum. Über alle Buchungen lag die Median-Lead-Time bei 38 Tagen, bei den hochpreisigen Buchungen dagegen schon bei 51 Tagen. Hier ist es also prinzipiell weniger wichtig, viele Zimmer vorzuhalten, da die gäste mit viel Geld sowieso etwas früher buchen.


In [ ]:
# ── Figure 4: Lead time by price bracket (50€ steps from Q3 threshold) ────────
ADR_STARTS = {"City Hotel": 127, "Resort Hotel": 120}
STEP = 50
MAX_ADR = 500

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, hotel in zip(axes, HOTELS):
    start = ADR_STARTS[hotel]
    edges = list(range(start, MAX_ADR + 1, STEP)) + [9999]
    labels = [f"€{edges[i]}–{edges[i+1]}" for i in range(len(edges) - 2)]
    labels.append(f"€{edges[-2]}+")

    sub = realised[(realised["hotel"] == hotel) & (realised["adr"] >= start)].copy()
    sub["price_bucket"] = pd.cut(sub["adr"], bins=edges, labels=labels,
                                  include_lowest=True, right=False)

    groups = [sub[sub["price_bucket"] == lbl]["lead_time"].dropna().values
              for lbl in labels]

    # Drop buckets with fewer than 5 observations
    valid = [(lbl, g) for lbl, g in zip(labels, groups) if len(g) >= 5]
    valid_labels, valid_groups = zip(*valid)

    bp = ax.boxplot(valid_groups, patch_artist=True,
                    medianprops=dict(color="black", lw=2.5),
                    whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2),
                    flierprops=dict(marker=".", markersize=2, alpha=0.25),
                    widths=0.55)
    for patch in bp["boxes"]:
        patch.set_facecolor(HOTEL_COLORS[hotel])
        patch.set_alpha(0.72)

    for i, g in enumerate(valid_groups):
        m = np.median(g)
        ax.text(i + 1, m + 3, f"{m:.0f}d", ha="center", va="bottom",
                fontsize=8, fontweight="bold", color="#333333")
        ax.text(i + 1, 2, f"n={len(g)}", ha="center", va="bottom",
                fontsize=7, color="gray")

    ax.set_xticklabels(valid_labels, rotation=30, ha="right")
    ax.set_ylim(0, 200)
    ax.set_title(hotel, fontweight="bold", fontsize=13)
    ax.set_xlabel("Preisklasse (ADR)")
    ax.set_ylabel("Lead time (Tage vor Ankunft)")

fig.suptitle("Figure 4 · Lead time nach Preisklasse (50€-Schritte)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("q1_fig4_leadtime_by_price.png", dpi=150, bbox_inches="tight")
plt.show()


**Abbildung 4 – Lead time nach Preisklasse**

Beim City-Hotel ist klar zu sehen, dass je teurer die Buchung ist, desto kürzer ist die Lead-Time. Die Lead-Time befindet sich hier im Schnitt zwischen 36 und 21 Tagen, was ziemlich kurzfristig ist. Beim Resorthotel ist ein ähnlicher Trend zu beobachten. Hier sinkt die Lead-Time von im Schnitt 59 Tagen auf 34 Tage. Einzig die teuersten Buchungen stellen einen Ausreißer dar. Sie werden recht früh gebucht, mit einer Lead-Time von im Schnitt 64 Tagen. Dieses Diagramm zeigt, wie wichtig es ist Zimmer für die hochpreisigen Kunden vorzuhalten. Wenn einem am Ende die Zimmer ausgehen, verpasst man die Kunden, die am meisten Zahlen würden.


In [ ]:
# ── Figure 5: Anteil hochpreisiger Spätbucher an allen Übernachtungen ──────────
FILTERS = {
    "City Hotel":   {"adr": 127, "lead_time": 50},
    "Resort Hotel": {"adr": 120, "lead_time": 38},
}

shares = {}
for hotel, f in FILTERS.items():
    total = len(realised[realised["hotel"] == hotel])
    high_late = len(realised[
        (realised["hotel"] == hotel) &
        (realised["adr"] > f["adr"]) &
        (realised["lead_time"] < f["lead_time"])
    ])
    shares[hotel] = high_late / total * 100

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(shares.keys(), shares.values(),
              color=[HOTEL_COLORS[h] for h in shares], alpha=0.85, width=0.5)

for bar, val in zip(bars, shares.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("Anteil an allen realisierten Übernachtungen (%)")
ax.set_title(
    "Figure 5 · Anteil hochpreisiger Spätbucher\n"
    "(City: ADR > €127 & Lead time < 50d | Resort: ADR > €120 & Lead time < 38d)",
    fontweight="bold", fontsize=12
)
plt.tight_layout()
plt.savefig("q1_fig5_highfare_latebooker_share.png", dpi=150, bbox_inches="tight")
plt.show()

for hotel, s in shares.items():
    print(f"{hotel}: {s:.1f}% aller realisierten Übernachtungen")


**Abbildung 5 – Hochpreisige Spätbucher pro Hotel**

Dieses Balkendiagramm zeigt, wie viele Zimmer pro Nacht pro Hotel für hochpreisige Buchungen geblockt werden sollten. Beim City-Hotel sind es 13,9%. Bei der Berechnung wurde wie folgt vorgegangen: Beim City-Hotel werden nur die 25% teuersten Buchungen gezählt, die eine Lead-Time von weniger als 50 Tagen hatten. Diese Anzahl an Buchungen wird dann durch die Gesamtzahl an Buchungen geteilt. Die Median-Lead-Time über alle Buchungen im City-Hotel liegt bei 50. Schützen muss man nur Buchungen, die eine geringere Lead-Time haben als die 50 Tage. Davor ist davon auszugehen, dass sowieso noch genug Zimmer verfügbar sind. Und der ADR-Preis ist auf das 75% Quantil gelegt worden, um nur teure Buchungen zu schützen. Würde er niedriger liegen, dann müsste man für mittelteure Buchungen ein zu hohes Risiko eingehen. 

Beim Resort-Hotel liegt der Wert an zu schützenden Zimmern nur bei 10,6%. Auch hier wurde das 75% Quantil für die teuersten Buchungen gewählt. Außerdem mussten auch diese Buchungen das Kriterium einer Lead-Time von unter der Median-Lead-Time aller Buchungen erfüllen. In diesem Fall sind das 38 Tage.

Die beiden Prozentwerte spiegeln damit die tägliche Situation im jeweiligen Hotel wider. Im City-Hotel werden hochpreisige Buchungen eher später getätigt. Dies liegt zum Teil vermutlich an Geschäftsreisenden, die spontan buchen, aber nicht so sehr auf das Geld achten müssen. Beim Resort-Hotel müssen die spät Buchenden zwar auch mehr zahlen als die Frühbucher, allerdings wird hier generell früher gebucht. Auch teure Buchungen sind keine Last-Minute Buchungen, sondern gehen im Median schon 51 Tage vor der Übernachtung ein. Deshalb muss man im Resort-Hotel weniger Zimmer vorhalten, da die teuren Buchungen oft früh genug einhergehen, ohne das andere billige Buchungen ihnen die Zimmer wegnehmen.


## §Q2 · Derive Overbooking Parameters

Clara wants to overbook to mitigate late cancellations and no-shows — but walk-aways hurt guest satisfaction and brand image.

- Use historical data to estimate a **robust overbooking buffer** for each hotel. How many extra bookings can Clara accept without incurring frequent walk-aways?
- Make any **trade-offs and assumptions explicit** (walk cost, tolerated walk frequency, independence of cancellations, …).

*Methods reference: companion NB1 §7 (cancellation by channel / lead-time), NB3 §6 (cancellation model + Brier), §8.5 (buffer calculator with tolerated-walk slider).*

Wir wollen herausfinden, wie viele Überbuchungen in Prozent wir zulassen können, ohne das am Ende die Hotels tatsächlich mehr Gäste angenommen haben als sie aufnehmen können. Beim ausrechnen dieses Prozentwertes sind nur Tage relevant, an dem das Hotel auch tatsächlich ausgebucht war. Alle anderen Tage können wir ignorieren. Dafür müssen wir zuerst ausrechnen, was für eine Kapazität die Hotels überhaupt haben. Man könnte denken, der höchste Buchungswert stellt dann die maximale Kapazität dar. Dies ist aber falsch, da es sich hier sehr wahrscheinlich um eine Überbuchung handelt. Die echte Kapazität liegt irgendwo knapp darunter.

In [ ]:
# ── Figure 0: Tägliche Belegung pro Hotel ─────────────────────────────────────
# Jede Buchung wird auf alle ihre Übernachtungen ausgeweitet,
# damit wir für jeden Kalendertag die tatsächliche Belegung zählen können.

stay_df = realised[realised["total_nights"] > 0].copy()
stay_df["night_dates"] = stay_df.apply(
    lambda r: pd.date_range(r["arrival_date"], periods=int(r["total_nights"]), freq="D"),
    axis=1,
)
expanded = stay_df[["hotel", "night_dates"]].explode("night_dates")
daily_occ = (
    expanded.groupby(["hotel", "night_dates"])
    .size()
    .reset_index(name="overnight_stays")
)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

for ax, hotel in zip(axes, HOTELS):
    sub = daily_occ[daily_occ["hotel"] == hotel].sort_values("night_dates")
    ax.plot(sub["night_dates"], sub["overnight_stays"],
            color=HOTEL_COLORS[hotel], lw=0.8, alpha=0.9)
    ax.fill_between(sub["night_dates"], sub["overnight_stays"],
                    color=HOTEL_COLORS[hotel], alpha=0.18)

    peak = sub.loc[sub["overnight_stays"].idxmax()]
    ax.axhline(peak["overnight_stays"], color="red", ls="--", lw=1.0,
               label=f"Max: {peak['overnight_stays']} ({peak['night_dates'].date()})")

    ax.set_title(hotel, fontweight="bold", fontsize=13)
    ax.set_ylabel("Übernachtungen")
    ax.legend(fontsize=9)

axes[-1].set_xlabel("Datum")
fig.suptitle("Figure 0 · Tägliche Belegung — Anzahl tatsächlicher Übernachtungen pro Tag",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("q1_fig0_daily_occupancy.png", dpi=150, bbox_inches="tight")
plt.show()

Um jetzt die echte Kapazität herauszufinden, schauen wir nur Tage an, die 15% unter der maximal stattgefundenen Auslastung liegen. Von diesen Tagen nehmen wir das 80% Quantil, da wir davon ausgehen, dass in max. 20% der Fälle das Hotel knapp überbucht war und in der anderen Hälfte nicht komplett voll war. Somit haben wir für das Cityhotel eine Kapazität von 223 Zimmern und für das Resorthotel von 183 Zimmern.

In [ ]:
# ── Figure 0b: Kapazitätsschätzung aus Spitzentagen ──────────────────────────
# Spitzentage = Tage mit Belegung >= 85% des Maximums (0–15% darunter)
# Geschätzte Kapazität = Median dieser Spitzentage

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

CAPACITY = {}

for ax, hotel in zip(axes, HOTELS):
    occ = daily_occ[daily_occ["hotel"] == hotel]["overnight_stays"]
    max_occ = occ.max()
    threshold = max_occ * 0.85

    peak_days = occ[occ >= threshold].sort_values(ascending=False).reset_index(drop=True)
    capacity = peak_days.quantile(0.85)
    CAPACITY[hotel] = round(capacity)

    ax.scatter(range(len(peak_days)), peak_days,
               color=HOTEL_COLORS[hotel], alpha=0.75, s=25, zorder=3)
    ax.axhline(max_occ,  color="red",   ls="--", lw=1.5,
               label=f"Maximum: {max_occ}")
    ax.axhline(threshold, color="gray", ls=":",  lw=1.5,
               label=f"Schwelle (−15 %): {threshold:.0f}")
    ax.axhline(capacity,  color="black", ls="-", lw=2.2,
               label=f"Geschätzte Kapazität (85%-Quantil): {capacity:.0f}")

    ax.set_title(hotel, fontweight="bold", fontsize=13)
    ax.set_xlabel("Spitzentage (absteigend nach Belegung sortiert)")
    ax.set_ylabel("Übernachtungen")
    ax.legend(fontsize=9)

    print(f"{hotel}: Max={max_occ} | Schwelle={threshold:.0f} | "
          f"n Spitzentage={len(peak_days)} | Kapazität≈{CAPACITY[hotel]}")

fig.suptitle(
    "Figure 0b · Kapazitätsschätzung — Median der Tage im Bereich 0–15 % unter dem Maximum",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig("q1_fig0b_capacity_estimate.png", dpi=150, bbox_inches="tight")
plt.show()

Ab jetzt teilen wir die Buchungen in drei verschiedene Gruppen ein. Gruppe 1 sind Tage, an denen das Hotel zwischen >=95% Auslastung hat. In Gruppe 2 sind Tage mit einer Auslastung zwischen 80% und 95%. Gruppe drei beinhaltet den Rest mit einer Auslastung von unter 80%. Im weiteren Verlauf interessiert uns aber nur noch Gruppe 1 und 2.

In [ ]:
GROUPS = {
    "City Hotel": [
        ("Gruppe 1 (212–226)",  212, 226),
        ("Gruppe 2 (179–211)",  179, 211),
        ("Gruppe 3 (< 179)",      0, 178),
    ],
    "Resort Hotel": [
        ("Gruppe 1 (174–187)",  174, 187),
        ("Gruppe 2 (147–173)",  147, 173),
        ("Gruppe 3 (< 147)",      0, 146),
    ],
}

for hotel, groups in GROUPS.items():
    occ = daily_occ[daily_occ["hotel"] == hotel]["overnight_stays"]
    print(f"{hotel}")
    for label, lo, hi in groups:
        count = ((occ >= lo) & (occ <= hi)).sum()
        print(f"  {label}: {count} Tage")
    print()

Für Tage aus Gruppe 1 können wir uns anschauen, wie viele Stornierungen es als Anteil aller Buchungen gab.

In [ ]:
# ── Q2 Figure 1: Ø Stornierungsrate an Hochauslastungstagen ─────────────────
# Nur Tage mit tatsächlicher Belegung > 212 (City Hotel) bzw. > 174 (Resort Hotel)

THRESHOLDS_Q2 = {"City Hotel": 212, "Resort Hotel": 174}

cancel_rates = {}
for hotel, threshold in THRESHOLDS_Q2.items():
    high_occ_dates = daily_occ[
        (daily_occ["hotel"] == hotel) &
        (daily_occ["overnight_stays"] > threshold)
    ]["night_dates"]

    hotel_bookings = bookings[
        (bookings["hotel"] == hotel) &
        (bookings["arrival_date"].isin(high_occ_dates))
    ]

    daily_cancel_rate = (
        hotel_bookings.groupby("arrival_date")["is_canceled"]
        .mean() * 100
    )
    cancel_rates[hotel] = daily_cancel_rate.mean()
    print(f"{hotel}: {len(high_occ_dates)} qualifizierende Tage | "
          f"Ø Stornierungsrate = {cancel_rates[hotel]:.1f}%")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    cancel_rates.keys(), cancel_rates.values(),
    color=[HOTEL_COLORS[h] for h in cancel_rates],
    alpha=0.85, width=0.45
)

for bar, rate in zip(bars, cancel_rates.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{rate:.1f}%",
        ha="center", va="bottom", fontweight="bold", fontsize=12
    )

ax.set_ylabel("Ø Stornierungsrate an Hochauslastungstagen (%)", fontsize=11)
ax.set_title(
    "Q2 Figure 1 · Ø Stornierungsrate an Hochauslastungstagen\n"
    "(City Hotel: Belegung > 212 | Resort Hotel: Belegung > 174)",
    fontweight="bold", fontsize=12
)
ax.set_ylim(0, max(cancel_rates.values()) * 1.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("q2_fig1_cancel_rate_high_occ.png", dpi=150, bbox_inches="tight")
plt.show()

Wir schauen uns als nächstes an, wann diese Stornierungen stattfinden.

In [ ]:
# ── Kumulierter Stornierungsanteil nach Lead-time ────────────────────────────
# Nur Hochauslastungstage: City Hotel > 212, Resort Hotel > 174 Übernachtungen
# Bei Lead-time X: Anteil aller Buchungen, die noch >= X Tage vor Ankunft storniert wurden

THRESHOLDS_Q3 = {"City Hotel": 212, "Resort Hotel": 174}

fig, ax = plt.subplots(figsize=(14, 5))

for hotel, threshold in THRESHOLDS_Q3.items():
    high_occ_dates = daily_occ[
        (daily_occ["hotel"] == hotel) &
        (daily_occ["overnight_stays"] > threshold)
    ]["night_dates"]

    hotel_bookings_q3 = bookings[
        (bookings["hotel"] == hotel) &
        (bookings["arrival_date"].isin(high_occ_dates))
    ]
    total_bookings = len(hotel_bookings_q3)

    cancelled = hotel_bookings_q3[hotel_bookings_q3["is_canceled"] == 1]
    cancel_per_lt = (
        cancelled.groupby("lead_time").size()
        .reindex(range(366), fill_value=0)
    )

    # Kumulierung von rechts: bei X = alle Stornierungen mit lead_time >= X / alle Buchungen
    cumulative_pct = cancel_per_lt[::-1].cumsum()[::-1] / total_bookings * 100

    ax.plot(cumulative_pct.index, cumulative_pct.values,
            color=HOTEL_COLORS[hotel], lw=2.0, label=hotel)
    ax.fill_between(cumulative_pct.index, cumulative_pct.values,
                    color=HOTEL_COLORS[hotel], alpha=0.10)

    print(f"{hotel}: {total_bookings:,} Buchungen | "
          f"Gesamtstornierungsrate: {cancel_per_lt.sum() / total_bookings * 100:.1f}%")

ax.set_xlim(0, 365)
ax.set_xlabel("Lead time (Tage vor Ankunft)", fontsize=11)
ax.set_ylabel("Kumulierter Stornierungsanteil (%)", fontsize=11)
ax.set_title(
    "Kumulierter Stornierungsanteil nach Lead-time\n"
    "(Anteil aller Buchungen, die noch ≥ X Tage vor Ankunft storniert wurden | "
    "City > 212 | Resort > 174)",
    fontweight="bold", fontsize=12,
)
ax.legend(fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("q2_fig2_cancel_cumsum_pct.png", dpi=150, bbox_inches="tight")
plt.show()

Wir sehen, dass die Anzahl an Stornierungen ziemlich linear ohne größere Ausreißer ansteigt. Die beiden Endwerte von 29,7% und 38,9% können wir aber nicht einfach als Überbuchungsprozentsatz übernehmen. Ansonsten würde man das Hotel restlos überbuchen und müsste hohe Strafkosten bezahlen für Gäste, die kein freies Zimmer mehr bekommen haben. Frühe Stornierungen können wieder durch spätere Buchungen ausgeglichen werden. Daher zählen wir ab jetzt nur noch Stornierungen, die unter 20 Tage vor der Übernachtung eingegangen sind.

In [ ]:
# ── Kurzfristige Stornierungsrate an Hochauslastungstagen ─────────────────────
# Zähler:  Stornierungen mit lead_time < 20 an Hochauslastungstagen
# Nenner:  tatsächlich realisierte Übernachtungen an denselben Tagen

THRESHOLDS = {"City Hotel": 212, "Resort Hotel": 174}
LEADTIME_CUTOFF = 20

rates = {}
for hotel, threshold in THRESHOLDS.items():
    high_occ_dates = daily_occ[
        (daily_occ["hotel"] == hotel) &
        (daily_occ["overnight_stays"] > threshold)
    ]["night_dates"]

    # Zähler: Stornierungen mit lead_time < 20
    n_cancel_late = bookings[
        (bookings["hotel"] == hotel) &
        (bookings["arrival_date"].isin(high_occ_dates)) &
        (bookings["is_canceled"] == 1) &
        (bookings["lead_time"] < LEADTIME_CUTOFF)
    ].shape[0]

    # Nenner: realisierte Übernachtungen
    n_realised = realised[
        (realised["hotel"] == hotel) &
        (realised["arrival_date"].isin(high_occ_dates))
    ].shape[0]

    rates[hotel] = n_cancel_late / n_realised * 100
    print(f"{hotel}: {n_cancel_late} Spät-Stornierungen (LT < {LEADTIME_CUTOFF}) | "
          f"{n_realised} realisierte Übernachtungen | Rate = {rates[hotel]:.2f}%")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(rates.keys(), rates.values(),
              color=[HOTEL_COLORS[h] for h in rates], alpha=0.85, width=0.45)

for bar, val in zip(bars, rates.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.2f}%", ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("Kurzfristige Stornierungsrate (%)", fontsize=11)
ax.set_title(
    f"Kurzfristige Stornierungsrate an Hochauslastungstagen\n"
    f"(Stornierungen mit Lead time < {LEADTIME_CUTOFF} Tage / realisierte Übernachtungen | "
    f"City > 212 | Resort > 174)",
    fontweight="bold", fontsize=11,
)
ax.set_ylim(0, max(rates.values()) * 1.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("q2_fig3_late_cancel_rate.png", dpi=150, bbox_inches="tight")
plt.show()

Die beiden Balken zeigen an, wie viel Prozent an Stornierungen im Vergleich zu tatsächlich durchgeführten Buchungen es für Bereiche mit einer Lead-Time von weniger als 20 gab. Die beiden Werte, 4,8% und 4,1% stellen einen sinnvollen Wert für eine Überbuchung der Kapazitäten der Hotels dar. Da im Cityhotel mehr storniert wird, ist hier der Wert auch höher.

## §Q3 · Seasonally Update Your Rules

Demand patterns shift across the year. Clara wants Q1's advance sale limits and Q2's overbooking buffers to **adapt to seasonality**.

- Using appropriate aggregates from past years, identify periods of stronger or weaker demand **for each hotel**.
- Provide **updated rules** (per hotel × period) and justify each adjustment with observed patterns.

*Methods reference: companion NB1 §6 (temporal patterns), §9 with the month picker (re-run Q1's cut per season), NB3 §6 (re-fit or re-evaluate cancellation model per season if useful).*

In [ ]:
monthly_occ = (
    daily_occ.assign(month=lambda x: x["night_dates"].dt.month)
    .groupby(["hotel", "month"])["overnight_stays"]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14,5))

for ax, hotel in zip(axes, HOTELS):
    sub = monthly_occ[monthly_occ["hotel"] == hotel]

    ax.bar(
        sub["month"],
        sub["overnight_stays"],
        color=HOTEL_COLORS[hotel],
        alpha=0.8
    )

    avg = sub["overnight_stays"].mean()
    ax.axhline(avg, color="black", ls="--",
               label=f"Jahresmittel = {avg:.0f}")

    ax.set_title(hotel, fontweight="bold")
    ax.set_xlabel("Monat")
    ax.set_ylabel("Ø Übernachtungen")
    ax.legend()

plt.suptitle(
    "Q3 Figure 1 · Durchschnittliche Auslastung nach Monat",
    fontweight="bold"
)
plt.tight_layout()
plt.show()

### 1. Definition der saisonalen Nachfragephasen

Um Claras Regeln aus Q1 und Q2 präzise zu dynamisieren, reicht eine starre Betrachtung einzelner Monate nicht aus. Wir clustern das Jahr basierend auf der obigen durchschnittlichen Auslastung in drei strategische Saisonphasen pro Hotel:

* **High Season (Hauptsaison):** Monate, in denen die Auslastung weit über dem Jahresmittel liegt (City: Mai, Juni, September, Oktober | Resort: Juli, August).
* **Mid Season (Zwischensaison):** Monate mit stabiler, durchschnittlicher Nachfrage.
* **Low Season (Nebensaison):** Monate mit starkem Nachfrageeinbruch (Wintermonate; beim City-Hotel zusätzlich das typische "Sommerloch" im August durch fehlende Geschäftsreisen).

In [ ]:
# Definition der Saison-Zugehörigkeit per Hotel und Monat
def get_season(row):
    hotel = row["hotel"]
    month = row["arrival_date"].month
    
    if hotel == "City Hotel":
        if month in [5, 6, 9, 10]: return "High"
        elif month in [12, 1, 2, 8]: return "Low"
        else: return "Mid"
    elif hotel == "Resort Hotel":
        if month in [7, 8]: return "High"
        elif month in [11, 12, 1, 2]: return "Low"
        else: return "Mid"

# Saisons zuweisen (nur für realisierte Stays für Q1, Gesamtdaten für Q2)
realised_seasonal = realised.copy()
realised_seasonal["season"] = realised_seasonal.apply(get_season, axis=1)

bookings_seasonal = bookings.copy()
bookings_seasonal["season"] = bookings_seasonal.apply(get_season, axis=1)

print("=== UPDATE Q1: Saisonale Schutzkontingente für Spätbucher ===")
seasonal_q1 = {}
for hotel, f in FILTERS.items():
    seasonal_q1[hotel] = {}
    print(f"\n{hotel}:")
    for season in ["High", "Mid", "Low"]:
        sub_total = realised_seasonal[(realised_seasonal["hotel"] == hotel) & (realised_seasonal["season"] == season)]
        sub_high_late = sub_total[
            (sub_total["adr"] > f["adr"]) & 
            (sub_total["lead_time"] < f["lead_time"])
        ]
        share = (len(sub_high_late) / len(sub_total)) * 100 if len(sub_total) > 0 else 0
        seasonal_q1[hotel][season] = share
        print(f"  {season} Season: {share:.1f}% der Zimmer für teure Spätbucher schützen")

print("\n" + "="*60 + "\n")
print("=== UPDATE Q2: Saisonale Überbuchungspuffer (LT < 20 Tage) ===")
seasonal_q2 = {}
for hotel, threshold in THRESHOLDS.items():
    seasonal_q2[hotel] = {}
    print(f"\n{hotel}:")
    
    # Tage mit hoher Auslastung identifizieren
    high_occ_dates = daily_occ[(daily_occ["hotel"] == hotel) & (daily_occ["overnight_stays"] > threshold)]["night_dates"]
    
    for season in ["High", "Mid", "Low"]:
        # Nenner: Realisierte Buchungen an Hochauslastungstagen in dieser Saison
        n_realised = realised_seasonal[
            (realised_seasonal["hotel"] == hotel) & 
            (realised_seasonal["season"] == season) & 
            (realised_seasonal["arrival_date"].isin(high_occ_dates))
        ].shape[0]
        
        # Zähler: Kurzfristige Stornierungen (LT < 20) an Hochauslastungstagen in dieser Saison
        n_cancel_late = bookings_seasonal[
            (bookings_seasonal["hotel"] == hotel) & 
            (bookings_seasonal["season"] == season) & 
            (bookings_seasonal["arrival_date"].isin(high_occ_dates)) & 
            (bookings_seasonal["is_canceled"] == 1) & 
            (bookings_seasonal["lead_time"] < 20)
        ].shape[0]
        
        rate = (n_cancel_late / n_realised) * 100 if n_realised > 0 else 0
        seasonal_q2[hotel][season] = rate
        print(f"  {season} Season: Kurzfrist-Stornorate = {rate:.2f}% -> Empfohlener Puffer: {round(rate):.0f}%")

### 2. Strategische Begründung & operative Entscheidungs-Matrix

Die datengetriebene Zerlegung zeigt deutliche Verschiebungen, die Clara in der Praxis nutzen muss:

1.  **Das Resort-Hotel in der High Season (Juli/August):** Hier sinkt die kurzfristige Stornierungsrate spürbar. Familien stornieren den lang geplanten Sommerurlaub selten spontan. Würde Clara hier den Jahres-Pauschalwert von 4,1% anwenden, käme es unweigerlich zu teuren und rufschädigenden Abweisungen ("Walk-aways"). Der Puffer muss zwingend auf **3%** gesenkt werden. Gleichzeitig steigt der Anteil hochpreisiger Spätbucher auf **12,4%**, weshalb das Frühbucherkontingent (*Advance Sale Limit*) im Sommer knapper gehalten werden muss.
2.  **Das City-Hotel in der High Season (Mai/Juni/Sept/Okt):**
    Getrieben durch das Messe- und Kongressgeschäft steigt der Anteil hochpreisiger, kurzfristiger Buchungen massiv an (**15,5%**). Hier müssen deutlich mehr Zimmer für die lukrative Spätphase geschützt werden. Da Geschäftsreisende jedoch auch flexibler stornieren, bleibt die Spät-Stornierungsrate mit **4,9%** hoch – der Überbuchungspuffer darf hier mutig gewählt werden.

#### Finale Entscheidungs-Matrix für das Revenue-Management Team:

| Hotel | Saisonphase | Zimmerkapazität (berechnet) | Schutzkontingent Spätbucher (Zimmer) | Max. erlaubte Überbuchung (Zimmer / %) |
| :--- | :--- | :--- | :--- | :--- |
| **City Hotel** | High Season | 223 Zimmer | **35 Zimmer** (~15.5%) | **+11 Zimmer** (~5%) |
| | Mid Season | 223 Zimmer | **30 Zimmer** (~13.5%) | **+11 Zimmer** (~5%) |
| | Low Season | 223 Zimmer | **28 Zimmer** (~12.5%) | **+9 Zimmer** (~4%) |
| **Resort Hotel**| High Season | 183 Zimmer | **23 Zimmer** (~12.4%) | **+5 Zimmer** (~3%) |
| | Mid Season | 183 Zimmer | **20 Zimmer** (~10.8%) | **+7 Zimmer** (~4%) |
| | Low Season | 183 Zimmer | **13 Zimmer** (~7.1%) | **+9 Zimmer** (~5%) |

*Hinweis für das operative Team: Das Advance Sale Limit (Frühbucher-Kontingent) ergibt sich jeweils aus der Restkapazität abzüglich des Schutzkontingents und zuzüglich des erlaubten Überbuchungspuffers.*

### Saisonale Nachfragephasen identifizieren

Bevor die Regeln aus Q1 und Q2 saisonal angepasst werden können, müssen zunächst Perioden mit hoher und niedriger Nachfrage identifiziert werden. Dafür wird die durchschnittliche tägliche Auslastung pro Monat berechnet. Anstatt die Saisonen manuell festzulegen, werden die Monate anhand ihrer historischen Auslastung automatisch in High-, Shoulder- und Low-Season eingeteilt. Dadurch basiert die spätere Preis- und Kapazitätssteuerung vollständig auf beobachteten Nachfrageunterschieden.

In [ ]:
# Durchschnittliche Auslastung pro Monat

monthly_occ = (
    daily_occ.assign(month=lambda x: x["night_dates"].dt.month)
    .groupby(["hotel", "month"])["overnight_stays"]
    .mean()
    .reset_index()
)

season_tables = {}

for hotel in HOTELS:

    sub = monthly_occ[monthly_occ["hotel"] == hotel].copy()

    q33 = sub["overnight_stays"].quantile(0.33)
    q66 = sub["overnight_stays"].quantile(0.66)

    sub["season"] = np.where(
        sub["overnight_stays"] >= q66,
        "High",
        np.where(
            sub["overnight_stays"] >= q33,
            "Shoulder",
            "Low"
        )
    )

    season_tables[hotel] = sub

    print("\n", hotel)
    print(sub[["month", "overnight_stays", "season"]])

### Saisonale Advance-Sale-Limits berechnen

In Q1 wurde eine jährliche Schutzquote für hochpreisige Spätbucher bestimmt. Da sich die Nachfrage jedoch im Jahresverlauf verändert, wird diese Quote nun für jede Saison separat berechnet. Als schützenswerte Gäste gelten weiterhin Buchungen im obersten ADR-Quartil, die später als der Median aller Buchungen eingehen. Der resultierende Anteil gibt an, wie viele Zimmer in der jeweiligen Saison für potenziell profitablere Spätbucher reserviert werden sollten.

In [ ]:
ADVANCE_LIMITS = []

for hotel in HOTELS:

    hotel_data = realised[realised["hotel"] == hotel].copy()

    adr_cut = hotel_data["adr"].quantile(0.75)
    lt_cut = hotel_data["lead_time"].median()

    season_map = (
        season_tables[hotel]
        .set_index("month")["season"]
        .to_dict()
    )

    hotel_data["season"] = (
        hotel_data["arrival_date"]
        .dt.month
        .map(season_map)
    )

    for season in ["Low", "Shoulder", "High"]:

        sub = hotel_data[hotel_data["season"] == season]

        protected_share = (
            (
                (sub["adr"] > adr_cut) &
                (sub["lead_time"] < lt_cut)
            )
            .mean()
            * 100
        )

        ADVANCE_LIMITS.append([
            hotel,
            season,
            protected_share
        ])

advance_df = pd.DataFrame(
    ADVANCE_LIMITS,
    columns=["hotel", "season", "protected_pct"]
)

advance_df

Die Tabelle zeigt den Anteil hochpreisiger Spätbucher innerhalb jeder Saison. Höhere Werte deuten darauf hin, dass ein größerer Teil der Nachfrage erst relativ spät eintrifft und deshalb durch Kapazitätsschutz abgesichert werden sollte. Niedrigere Werte sprechen dagegen für eine stärkere Öffnung des Hotels für Frühbucherraten.

### Saisonale Overbooking-Puffer bestimmen

Neben der Kapazitätssteuerung muss auch die Überbuchungsstrategie an saisonale Unterschiede angepasst werden. Hierzu wird für jede Saison untersucht, wie häufig kurzfristige Stornierungen auftreten. Wie bereits in Q2 werden nur Stornierungen mit weniger als 20 Tagen Vorlauf betrachtet, da frühere Stornierungen meist durch neue Buchungen ersetzt werden können. Die daraus berechneten Quoten dienen als Grundlage für saisonale Overbooking-Puffer.

In [ ]:
monthly_late_cancel = (
    bookings
    .assign(month=lambda x: x["arrival_date"].dt.month)
    .groupby(["hotel","month"])
    .apply(
        lambda g: (
            (
                (g["is_canceled"] == 1) &
                (g["lead_time"] < 20)
            ).mean()
        ) * 100
    )
    .reset_index(name="late_cancel_rate")
)

In [ ]:
LEADTIME_CUTOFF = 20

OVERBOOK = []

for hotel in HOTELS:

    # Saison-Zuordnung aus Schritt 2
    season_map = (
        season_tables[hotel]
        .set_index("month")["season"]
        .to_dict()
    )

    # Alle Buchungen des Hotels
    hotel_bookings = bookings[
        bookings["hotel"] == hotel
    ].copy()

    hotel_bookings["season"] = (
        hotel_bookings["arrival_date"]
        .dt.month
        .map(season_map)
    )

    # Nur realisierte Aufenthalte
    hotel_realised = realised[
        realised["hotel"] == hotel
    ].copy()

    hotel_realised["season"] = (
        hotel_realised["arrival_date"]
        .dt.month
        .map(season_map)
    )

    for season in ["Low", "Shoulder", "High"]:

        # Kurzfristige Stornierungen (< 20 Tage)
        late_cancel = hotel_bookings[
            (hotel_bookings["season"] == season) &
            (hotel_bookings["is_canceled"] == 1) &
            (hotel_bookings["lead_time"] < LEADTIME_CUTOFF)
        ]

        # Realisierte Aufenthalte derselben Saison
        realised_stays = hotel_realised[
            hotel_realised["season"] == season
        ]

        # Saisonaler Overbooking-Puffer
        buffer = (
            len(late_cancel)
            / max(len(realised_stays), 1)
            * 100
        )

        OVERBOOK.append([
            hotel,
            season,
            buffer
        ])

overbook_df = pd.DataFrame(
    OVERBOOK,
    columns=[
        "hotel",
        "season",
        "overbooking_pct"
    ]
)

overbook_df["overbooking_pct"] = (
    overbook_df["overbooking_pct"]
    .round(2)
)

print(overbook_df)

### Saisonale Overbooking-Puffer bestimmen

Die in Q2 entwickelte Overbooking-Strategie wird nun auf die einzelnen Nachfragesaisonen übertragen. Dabei werden nur kurzfristige Stornierungen mit weniger als 20 Tagen Vorlauf berücksichtigt, da frühere Stornierungen in der Regel noch durch neue Buchungen ersetzt werden können. Für jede Saison wird die Anzahl dieser kurzfristigen Ausfälle ins Verhältnis zu den tatsächlich realisierten Aufenthalten gesetzt.

Die resultierenden Werte stellen saisonale Overbooking-Puffer dar. Höhere Werte deuten auf ein größeres Risiko kurzfristiger Kapazitätsverluste hin und rechtfertigen daher eine aggressivere Überbuchungsstrategie. Niedrigere Werte sprechen dagegen für ein vorsichtigeres Vorgehen, um Walk-Aways und die damit verbundenen Kosten zu vermeiden.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, hotel in zip(axes, HOTELS):

    sub = overbook_df[
        overbook_df["hotel"] == hotel
    ]

    ax.bar(
        sub["season"],
        sub["overbooking_pct"],
        color=HOTEL_COLORS[hotel],
        alpha=0.85
    )

    for _, row in sub.iterrows():
        ax.text(
            row["season"],
            row["overbooking_pct"] + 0.05,
            f"{row['overbooking_pct']:.2f}%",
            ha="center",
            fontweight="bold"
        )

    ax.set_title(hotel, fontweight="bold")
    ax.set_ylabel("Overbooking Buffer (%)")
    ax.set_xlabel("Season")

plt.suptitle(
    "Q3 Figure 2 · Seasonal Overbooking Buffers",
    fontweight="bold"
)

plt.tight_layout()
plt.show()

**Abbildung X – Saisonale Overbooking-Puffer**

Die Abbildung zeigt die geschätzten Overbooking-Puffer für jede Saison und jedes Hotel. Die Unterschiede zwischen den Saisonen spiegeln die unterschiedliche Häufigkeit kurzfristiger Stornierungen wider. Saisonen mit höheren Werten erlauben eine stärkere Überbuchung, da die Wahrscheinlichkeit freibleibender Zimmer größer ist. In Saisonen mit niedrigen Werten sollte dagegen konservativer geplant werden, um das Risiko von Überbelegungen und Gästeabwanderungen zu reduzieren.

### Saisonale Revenue-Management-Regeln zusammenführen

Im nächsten Schritt werden die Ergebnisse aus der Kapazitätssteuerung und der Überbuchungsanalyse kombiniert. Dadurch entsteht für jedes Hotel und jede Saison eine vollständige Revenue-Management-Regel. Die Tabelle bildet die zentrale Entscheidungsgrundlage für das operative Management während der Hoch- und Nebensaison.

In [ ]:
policy = (
    advance_df
    .merge(
        overbook_df,
        on=["hotel", "season"]
    )
    .sort_values(["hotel", "season"])
)

policy["protected_pct"] = (
    policy["protected_pct"]
    .round(1)
)

policy["overbooking_pct"] = (
    policy["overbooking_pct"]
    .round(1)
)

policy

Die finale Policy-Tabelle fasst die empfohlenen Maßnahmen für jede Saison zusammen. Die Schutzquote gibt an, welcher Anteil der Kapazität für hochpreisige Spätbucher reserviert werden sollte, während der Overbooking-Puffer den Umgang mit erwarteten kurzfristigen Stornierungen beschreibt.

### Visualisierung der saisonalen Regeln

Zur besseren Vergleichbarkeit werden die Ergebnisse zusätzlich grafisch dargestellt. Die Heatmap ermöglicht einen schnellen Überblick darüber, wie sich die empfohlenen Schutzquoten und Overbooking-Puffer zwischen den Saisonen unterscheiden. Dunklere Farben kennzeichnen jeweils höhere empfohlene Werte.

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12,5))

for ax, hotel in zip(axes, HOTELS):

    sub = policy[
        policy["hotel"] == hotel
    ].set_index("season")

    heat = sub[
        ["protected_pct", "overbooking_pct"]
    ]

    sns.heatmap(
        heat,
        annot=True,
        fmt=".1f",
        cmap="YlOrRd",
        ax=ax
    )

    ax.set_title(hotel)

plt.suptitle(
    "Q3 Final Policy Matrix",
    fontweight="bold"
)

plt.tight_layout()
plt.show()

**Abbildung X – Saisonale Revenue-Management-Matrix**

Die Heatmap verdeutlicht die saisonalen Unterschiede der empfohlenen Steuerungsgrößen. Insbesondere in Hochsaison-Perioden steigen die Schutzquoten für hochpreisige Spätbucher, während die Overbooking-Puffer tendenziell konservativer ausfallen. In nachfrageschwachen Monaten kann dagegen aggressiver überbucht und ein größerer Teil der Kapazität für Frühbucher freigegeben werden.

## Management-Fazit

Die saisonale Analyse zeigt, dass die Jahreswerte aus Q1 und Q2 nicht für alle Monate gleichermaßen geeignet sind. Während der Hochsaison sollte Clara insbesondere im City Hotel mehr Kapazität für hochpreisige Spätbucher schützen, da diese überdurchschnittlich kurzfristig buchen. Gleichzeitig sollte das Overbooking in nachfragestarken Monaten reduziert werden, um Walk-Aways auf nahezu ausgebuchten Tagen zu vermeiden.

In nachfrageschwachen Monaten kann die Strategie umgekehrt werden: geringere Schutzkontingente für Premiumgäste und höhere Overbooking-Puffer helfen dabei, freie Zimmer zu vermeiden und die Auslastung zu stabilisieren. Dadurch entsteht eine adaptive Revenue-Management-Politik, die sowohl den Umsatz als auch die operative Stabilität verbessert.

## §Q4 · Operate on Live Booking Data

Simulate a realistic decision environment for Viador:

1. **Pick a stay date** and extract its booking strip — every booking with any lead time that ultimately stays on that night.
2. Replay it as a **time-ordered stream** of requests (by lead time).
3. Implement an **adaptive control strategy** that:
    - Makes daily decisions based on current capacity and expected high-/low-fare arrivals.
    - **Dynamically updates** protection limits or overbooking thresholds as bookings (and cancellations) arrive.
    - **Integrates at least one trained model** (cancellation probability, ADR forecast, …) to improve decisions.

Report realised revenue, spillage, and walk-aways; compare against the static Q1 + Q2 baseline. A good recommendation is *transparent, robust to noise, and clearly communicated*.

*Methods reference: companion NB2 §3 (strip extraction), §4a (revenue surface), §4b (drill-down), §5 (S-index), NB3 §6 (calibrated model), §8 ($q^{\text{eff}}_t$ trajectory).*

In [ ]:
# Your Q4 analysis cells go here.

## §R · Reflection & methodology

Briefly answer (one short paragraph each):

1. **What is the single most important assumption** your policy relies on, and what would change if that assumption broke?
2. **What did you simplify** that a production deployment could not (cancellation release mechanics, λ stationarity, channel-mix drift, …)?
3. **What would you do with one more week** of effort?

*Your reflection answers go here.*